# Untranslatability inside a large model

A self-contained Colab notebook for the most targeted experiments in the untranslatables project, run on a **70B-class model**. It has four parts. Each saves its results to Google Drive and skips itself if they already exist, so a dropped session can be resumed by re-running every cell. (Part C's large covariance files live on the runtime's local disk, so after a full disconnect Part C recollects them, which takes about 30 minutes.)

| part | question | rough time on an A100 80 GB |
|---|---|---|
| **A. English detour** | When the model translates Spanish → Italian, does its working answer pass through English in the middle layers? (Wendler et al. 2024 on a bigger model) | 3 min |
| **B. Round trip, fixed** | How much of each word's meaning can be recovered from the translation? Source words in the translation are masked, so copying no longer counts. Borrowing is reported separately, and a sense-recovery score tests the "*duende* → elf" problem. | 15–25 min |
| **C. Round-trip spectrum** | Fit the best linear maps Spanish → English → Spanish between the model's own representations. The squared canonical correlations are the decay rates, so each word gets a round-trip retention, a half-life and an untranslatability U. English bias is read off as an asymmetry. Spanish–English and Hindi–English. | 50–80 min |
| **D. Causal mediation** | Split each word's state into the part shared with English and the private part. Does knocking out the private part hurt the model's *understanding* more than its *translation*? Does patching it flip *el cura* ↔ *la cura* in the English? | 10–15 min |

**GPU:** choose **A100 (80 GB)**, about 7.5 compute units an hour. The 4-bit 70B model needs about 40 GB, so the 40 GB A100, L4 and T4 are too small. An H100 or G4 (96 GB) also works if it is offered, and is faster per hour but costs more units. The whole notebook should take about 2–2.5 hours, i.e. roughly 15–20 units.

**Before spending A100 time:** set `MODEL_ID` to the 0.5B dry-run model in the config cell and run everything on a cheap T4 (about 10 minutes, ~0.2 units). It exercises every code path and catches errors before the expensive run.

Every number here is *inside one model* and, for Part C, *relative to the translation conventions in the training corpus*. The Hindi sentences are drafts awaiting a native speaker's check.

In [ ]:
# Runtime → Change runtime type → A100 GPU (High-RAM is selected automatically).
%pip install -q -U transformers accelerate bitsandbytes datasets spacy scikit-learn
!python -m spacy download es_core_news_sm -q

In [ ]:
# ============================ CONFIG ============================
MODEL_ID = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit"      # main run (pre-quantised 4-bit, ~40 GB download, ungated)
# MODEL_ID = "unsloth/Qwen2.5-72B-Instruct-bnb-4bit"      # alternative 70B-class model, stronger on Hindi
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"                 # DRY RUN on a T4 first

DRY_RUN = any(s in MODEL_ID for s in ["0.5B", "1.5B", "tiny"])
OUT_ROOT = "/content/drive/MyDrive/untranslatables_colab"   # results land here, one folder per model
LOCAL_CACHE = "/content/cache"                              # big intermediate files (covariances); not synced to Drive

RUN_A, RUN_B, RUN_C, RUN_D = True, True, True, True
PAIRS = ["es", "hi"]                    # Part C language pairs with English; drop "hi" to save ~25 min
N_SENT_FIT = 2000 if DRY_RUN else 30000 # parallel sentences used to fit the maps (per pair)
N_SENT_HELD = 200 if DRY_RUN else 1200  # held-out parallel sentences (never used for fitting)
LAMBDAS = [1e-3, 1e-2, 1e-1]            # ridge strengths tried for CCA (relative to mean variance)
SEED = 0
CHAT_DATE = "26 Jul 2024"               # pins Llama's date stamp so prompts are reproducible

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Only needed for gated models (the unsloth 4-bit checkpoints are not gated):
# from huggingface_hub import login; login()

In [ ]:
import os, re, json, math, time, random, difflib, gc
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt

def slug(s): return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
OUT = os.path.join(OUT_ROOT, slug(MODEL_ID)); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
CACHE = os.path.join(LOCAL_CACHE, slug(MODEL_ID)); os.makedirs(CACHE, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def _np(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, torch.Tensor): return o.tolist()
    raise TypeError(type(o))
def save_json(name, obj):
    with open(os.path.join(OUT, name), "w") as f: json.dump(obj, f, indent=1, default=_np, ensure_ascii=False)
def load_json(name):
    with open(os.path.join(OUT, name)) as f: return json.load(f)
def have(name): return os.path.exists(os.path.join(OUT, name))

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"▶ {self.label} ..."); return self
    def __exit__(self, *a): print(f"✓ {self.label}: {time.time()-self.t:.0f}s")

def auroc(pos, neg):
    """Probability that a random positive scores above a random negative (ties count half)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[~np.isnan(pos)], neg[~np.isnan(neg)]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    allv = np.concatenate([pos, neg]); r = rankdata(allv)
    return float((r[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg)))
def rankdata(a):
    a = np.asarray(a, float); order = np.argsort(a, kind="mergesort"); r = np.empty(len(a))
    sa = a[order]; i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j+1] == sa[i]: j += 1
        r[order[i:j+1]] = (i + j) / 2 + 1; i = j + 1
    return r
def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 3: return float("nan")
    return float(np.corrcoef(rankdata(x[m]), rankdata(y[m]))[0, 1])
def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed); m = rng.choice(x, (n, len(x))).mean(1)
    return (float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)))

# Visual language shared with the earlier figures
EASY, HARD, LOAN, GREY, INK = "#2f7d6d", "#c4572e", "#5b4fc4", "#d9d6cf", "#2b2b2b"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
def savefig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG, name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close(fig)
print("results ->", OUT)

## Test material
Invented sentences (no copyrighted text). `hot` maps each hard span to the sense meant *in this sentence*: this is what the sense-recovery score checks the translation against. The Hindi items are **drafts** for a native speaker to check.

In [ ]:
LANG_NAME = {"es": "Spanish", "pt": "Portuguese", "it": "Italian", "de": "German", "hi": "Hindi", "en": "English"}

SENTENCES = [
    # --- untranslatable words in context ---
    dict(id="duende", lang="es", cat="untranslatable", text="Me gusta mucho el duende.",
         hot={"duende": "the mysterious, almost demonic power and emotional intensity of a great artistic performance, as in flamenco (Lorca's sense)",
              "gusta": "it pleases me: the thing is the subject and I am the one affected, like having a taste for it"}),
    dict(id="sobremesa", lang="es", cat="untranslatable", text="Después de comer nos quedamos dos horas de sobremesa.",
         hot={"sobremesa": "the relaxed time spent talking at the table after a meal has finished"}),
    dict(id="madrugada", lang="es", cat="untranslatable", text="Llegamos a casa de madrugada, muertos de sueño.",
         hot={"madrugada": "the small hours, between midnight and dawn"}),
    dict(id="estrenar", lang="es", cat="untranslatable", text="Hoy estreno los zapatos que me regalaste.",
         hot={"estreno": "I am using or wearing something for the very first time"}),
    dict(id="verguenza", lang="es", cat="untranslatable", text="Sentí vergüenza ajena durante todo su discurso.",
         hot={"vergüenza ajena": "embarrassment felt on behalf of someone else who is embarrassing themselves"}),
    dict(id="tertulia", lang="es", cat="untranslatable", text="Los jueves vamos a la tertulia del café de la esquina.",
         hot={"tertulia": "a regular informal gathering of people for conversation, often literary or political"}),
    # --- constructions ---
    dict(id="olvido", lang="es", cat="construction", text="Se me olvidó la llave en el coche.",
         hot={"Se me olvidó": "I forgot it, but framed as something that happened to me by accident, as if the key forgot itself on me"}),
    dict(id="estar_aburrido", lang="es", cat="construction", text="Juan está aburrido en la reunión.",
         hot={"está aburrido": "is bored, a temporary state (not 'is boring')"}),
    dict(id="ser_aburrido", lang="es", cat="construction", text="Juan es aburrido en las reuniones.",
         hot={"es aburrido": "is boring, a lasting characteristic (not 'is bored')"}),
    dict(id="ojala", lang="es", cat="construction", text="Ojalá llueva mañana por la tarde.",
         hot={"Ojalá": "a strong hope or wish, 'if only', originally 'may God will it'"}),
    # --- idioms ---
    dict(id="pelo", lang="es", cat="idiom", text="Creo que me estás tomando el pelo.",
         hot={"tomando el pelo": "teasing me, pulling my leg"}),
    dict(id="nubes", lang="es", cat="idiom", text="Mi hermana siempre está en las nubes.",
         hot={"en las nubes": "daydreaming, absent-minded"}),
    dict(id="pelos", lang="es", cat="idiom", text="Mi abuela no tiene pelos en la lengua.",
         hot={"no tiene pelos en la lengua": "speaks very frankly, without holding anything back"}),
    # --- controls ---
    dict(id="tren", lang="es", cat="control", text="El tren sale a las ocho de la mañana.", hot={}),
    dict(id="hermano", lang="es", cat="control", text="Mi hermano vive en un piso pequeño.", hot={}),
    dict(id="pan", lang="es", cat="control", text="Compré pan y leche en la tienda.", hot={}),
    dict(id="reunion", lang="es", cat="control", text="La reunión empieza a las diez en punto.", hot={}),
    dict(id="gatos", lang="es", cat="control", text="Tengo dos gatos negros y un perro.", hot={}),
    dict(id="invierno", lang="es", cat="control", text="En invierno hace mucho frío en el norte.", hot={}),
    # --- other European languages (Part B only) ---
    dict(id="saudade", lang="pt", cat="untranslatable", text="Tenho muita saudade da minha avó.",
         hot={"saudade": "a deep, bittersweet longing for someone or something absent, that may never return"}),
    dict(id="magari", lang="it", cat="construction", text="Magari fosse vero!",
         hot={"Magari": "if only (a wistful wish)"}),
    dict(id="fernweh", lang="de", cat="untranslatable", text="Im Winter habe ich immer Fernweh.",
         hot={"Fernweh": "an ache to travel to faraway places, the opposite of homesickness"}),
    # --- Hindi (DRAFTS: to be checked by a native speaker) ---
    dict(id="hi_pasand", lang="hi", cat="construction", draft=True, text="मुझे चाय बहुत पसंद है।",
         hot={"मुझे": "to me (dative experiencer: the tea is liked by me, like Spanish 'me gusta')"}),
    dict(id="hi_jugaad", lang="hi", cat="untranslatable", draft=True, text="उसने जुगाड़ से पुरानी गाड़ी ठीक कर दी।",
         hot={"जुगाड़": "a clever improvised fix made with whatever is at hand"}),
    dict(id="hi_mann", lang="hi", cat="untranslatable", draft=True, text="आज पढ़ाई में मेरा मन नहीं लग रहा।",
         hot={"मन": "heart-mind: the seat of feeling, will and attention together"}),
    dict(id="hi_apnapan", lang="hi", cat="untranslatable", draft=True, text="इस घर में मुझे बहुत अपनापन लगता है।",
         hot={"अपनापन": "a warm sense of belonging and intimate familiarity, of being among one's own"}),
    dict(id="hi_train", lang="hi", cat="control", draft=True, text="ट्रेन सुबह आठ बजे चलती है।", hot={}),
    dict(id="hi_bhai", lang="hi", cat="control", draft=True, text="मेरा भाई दिल्ली में रहता है।", hot={}),
]

# Sense minimal pairs for Part D: same word, two contexts, two senses.
# The English candidates differ only in the critical word, so their log-odds measures which sense the translation carries.
SENSE_PAIRS = [
    dict(id="cura", word="cura", a="Me preocupa el cura.", b="Me preocupa la cura.",
         en_a="I am worried about the {}.", en_b="I am worried about the {}.", sense_a="priest", sense_b="cure"),
    dict(id="cometa", word="cometa", a="Miramos el cometa.", b="Miramos la cometa.",
         en_a="We looked at the {}.", en_b="We looked at the {}.", sense_a="comet", sense_b="kite"),
    dict(id="pendiente", word="pendiente", a="Vi el pendiente.", b="Vi la pendiente.",
         en_a="I saw the {}.", en_b="I saw the {}.", sense_a="earring", sense_b="slope"),
    dict(id="corte", word="corte", a="Me sorprendió el corte.", b="Me sorprendió la corte.",
         en_a="The {} surprised me.", en_b="The {} surprised me.", sense_a="cut", sense_b="court"),
    dict(id="frente", word="frente", a="Señaló el frente.", b="Señaló la frente.",
         en_a="He pointed at the {}.", en_b="He pointed at the {}.", sense_a="front", sense_b="forehead"),
    dict(id="papa", word="papa", a="Me gusta el papa.", b="Me gusta la papa.",
         en_a="I like the {}.", en_b="I like the {}.", sense_a="pope", sense_b="potato"),
    dict(id="aburrido", word="aburrido", a="Juan es aburrido.", b="Juan está aburrido.",
         en_a="Juan is {}.", en_b="Juan is {}.", sense_a="boring", sense_b="bored"),
    dict(id="duende_ctx", word="duende", a="En ese cuento infantil, lo que más me gusta es el duende.",
         b="En ese cante flamenco, lo que más me gusta es el duende.",
         en_a="In that children's story, what I like most is the {}.",
         en_b="In that flamenco song, what I like most is the {}.", sense_a="goblin", sense_b="soul"),
]

SID = {S["id"]: S for S in SENTENCES}

# Part A: Spanish -> Italian word translation, with the English equivalent for the logit lens
DETOUR_SHOTS = [("perro", "cane"), ("ventana", "finestra"), ("mesa", "tavolo"), ("leche", "latte")]
DETOUR_WORDS = [("árbol", "albero", "tree"), ("pájaro", "uccello", "bird"), ("manzana", "mela", "apple"),
    ("cuchillo", "coltello", "knife"), ("calle", "strada", "street"), ("niño", "bambino", "child"),
    ("nube", "nuvola", "cloud"), ("queso", "formaggio", "cheese"), ("llave", "chiave", "key"),
    ("zapato", "scarpa", "shoe"), ("camisa", "camicia", "shirt"), ("mantequilla", "burro", "butter"),
    ("cerdo", "maiale", "pig"), ("caballo", "cavallo", "horse"), ("hermano", "fratello", "brother"),
    ("pescado", "pesce", "fish"), ("cuchara", "cucchiaio", "spoon"), ("tenedor", "forchetta", "fork"),
    ("silla", "sedia", "chair"), ("puerta", "porta", "door"), ("ciudad", "città", "city"),
    ("rodilla", "ginocchio", "knee"), ("hielo", "ghiaccio", "ice"), ("lluvia", "pioggia", "rain"),
    ("nieve", "neve", "snow"), ("hoja", "foglia", "leaf"), ("lengua", "lingua", "tongue"),
    ("oreja", "orecchio", "ear"), ("huevo", "uovo", "egg"), ("ajo", "aglio", "garlic"),
    ("cerebro", "cervello", "brain"), ("espejo", "specchio", "mirror")]

## Load the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
with Timer(f"load {MODEL_ID}"):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    _dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=_dtype, device_map="auto")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=_dtype, device_map="auto")
    model.eval()

In [ ]:
def _get_attr_path(obj, path):
    for p in path.split("."): obj = getattr(obj, p)
    return obj
def find_decoder(m):
    for base in ["model", "model.model", "model.language_model", "transformer"]:
        try:
            b = _get_attr_path(m, base)
            layers = getattr(b, "layers", None) or getattr(b, "h", None)
            norm = getattr(b, "norm", None) or getattr(b, "ln_f", None)
            if layers is not None and norm is not None: return b, layers, norm
        except AttributeError: pass
    raise ValueError("could not find decoder layers")

BASE, LAYERS_MOD, FINAL_NORM = find_decoder(model)
LM_HEAD = model.get_output_embeddings()
N_LAYERS = len(LAYERS_MOD)
D_MODEL = model.config.hidden_size if hasattr(model.config, "hidden_size") else model.config.text_config.hidden_size
INPUT_DEV = model.get_input_embeddings().weight.device
BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
print(f"{N_LAYERS} layers, d={D_MODEL}, BOS={BOS}, input device {INPUT_DEV}")
if torch.cuda.is_available(): print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

WORD_RE = re.compile(r"[\wऀ-ॣ०-ॿ]+(?:['’]\w+)?")
def words_with_spans(text): return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(text)]
def enc(text, offsets=False): return tok(text, add_special_tokens=False, return_offsets_mapping=offsets)
def ntok(text): return len(enc(text)["input_ids"])

def chat_prompt(user, prefix=""):
    msgs = [{"role": "user", "content": user}]
    try: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string=CHAT_DATE)
    except Exception: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return p + prefix

def tokens_in_span(offsets, a, b):
    return [i for i, (x, y) in enumerate(offsets) if y > x and y > a and x < b]

def find_phrase(text, phrase):
    m = re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text, flags=re.I)
    return (m.start(), m.end()) if m else None

def positions_at(prompt, a, b):
    return tokens_in_span(enc(prompt, offsets=True)["offset_mapping"], a, b)

def positions_in_prompt(prompt, sentence, phrase=None, span=None):
    """Token positions of `phrase` (or of the character span `span`) inside `sentence`, inside `prompt`."""
    base = prompt.rfind(sentence); assert base >= 0, "sentence not in prompt"
    sp = span or find_phrase(sentence, phrase); assert sp is not None, f"{phrase!r} not in {sentence!r}"
    return positions_at(prompt, base + sp[0], base + sp[1])

class _Stop(Exception): pass

def make_edit_hook(edits_by_b):
    """edits_by_b: {batch index: [(positions LongTensor, fn)]}; fn maps states [k,d] -> new states [k,d]."""
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if h.shape[1] <= 1: return None              # cached decoding step: leave untouched
        for b, items in edits_by_b.items():
            for pos, fn in items:
                pos = pos.to(h.device)
                if int(pos.max()) < h.shape[1]:
                    h[b, pos] = fn(h[b, pos]).to(h.dtype)
        return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
    return hook

def _register_edits(edits_for_batch):
    """edits_for_batch: list (one per batch row) of lists of (layer, positions, fn)."""
    per_layer = defaultdict(lambda: defaultdict(list))
    for b, items in enumerate(edits_for_batch):
        for (layer, pos, fn) in (items or []):
            per_layer[layer][b].append((torch.as_tensor(pos, dtype=torch.long), fn))
    return [LAYERS_MOD[layer - 1].register_forward_hook(make_edit_hook(dict(eb))) for layer, eb in per_layer.items()]

@torch.no_grad()
def score(prompts, conts, batch=8, edits=None):
    """Teacher-forced log p of each continuation given its prompt.
    edits: optional list (per item) of [(layer, prompt token positions, fn)]; 'layer l' = hidden_states[l] = output of block l-1.
    Returns per item: dict(lp=array over continuation tokens, offsets=char offsets of those tokens in the continuation)."""
    res = [None] * len(prompts)
    for i in range(0, len(prompts), batch):
        idx = list(range(i, min(i + batch, len(prompts))))
        seqs, spans, offs = [], [], []
        for j in idx:
            p = enc(prompts[j])["input_ids"]; c = enc(conts[j], offsets=True)
            seqs.append(p + c["input_ids"]); spans.append((len(p), len(p) + len(c["input_ids"]))); offs.append(c["offset_mapping"])
        L = max(map(len, seqs))
        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        handles = _register_edits([edits[j] for j in idx]) if edits is not None else []
        try:
            out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV))
        finally:
            for h in handles: h.remove()
        H = out.last_hidden_state
        for b, j in enumerate(idx):
            s, e = spans[b]
            if e <= s: res[j] = dict(lp=np.zeros(0), offsets=[]); continue
            logits = LM_HEAD(H[b, s - 1:e - 1].to(LM_HEAD.weight.device)).float()
            tgt = ids[b, s:e].to(logits.device)
            res[j] = dict(lp=torch.log_softmax(logits, -1).gather(-1, tgt[:, None])[:, 0].cpu().numpy(), offsets=offs[b])
        del out, H
    return res

@torch.no_grad()
def generate(prompts, max_new_tokens=64, batch=16):
    tok.padding_side = "left"; outs = []
    for i in range(0, len(prompts), batch):
        e = tok(prompts[i:i + batch], add_special_tokens=False, return_tensors="pt", padding=True).to(INPUT_DEV)
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(r, skip_special_tokens=True).strip() for r in g[:, e["input_ids"].shape[1]:]]
    tok.padding_side = "right"
    return outs

@torch.no_grad()
def generate_with_edits(prompt, edits, max_new_tokens=40):
    """Greedy generation for one prompt, with edits applied to the prompt positions (prefill only)."""
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    handles = _register_edits([edits])
    try:
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(g[0, e["input_ids"].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def capture(ids, am, layers):
    """Residual states at `layers` (dict layer -> [B,T,d]); stops the forward pass after the deepest one."""
    store, deepest = {}, max(layers)
    def mk(l):
        def hook(m, i, o):
            store[l] = (o[0] if isinstance(o, tuple) else o).detach()
            if l == deepest: raise _Stop()
        return hook
    hs = [LAYERS_MOD[l - 1].register_forward_hook(mk(l)) for l in layers]
    try: BASE(input_ids=ids, attention_mask=am)
    except _Stop: pass
    finally:
        for h in hs: h.remove()
    return store

@torch.no_grad()
def capture_prompt_states(prompt, layer):
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    return capture(e["input_ids"], e["attention_mask"], [layer])[layer][0]

def encode_raw(text):
    e = enc(text, offsets=True)
    return BOS + e["input_ids"], [(0, 0)] * len(BOS) + list(e["offset_mapping"])

print("helpers ready")

## Part A: English detour (logit lens)
A few-shot Spanish → Italian word-translation prompt. At every layer the final position's state is read out through the final norm and unembedding. We track the probability of the Italian answer's first token against the English equivalent's. Prediction: English rises in the middle layers before Italian takes over.

In [ ]:
if RUN_A and not have("A_detour.json"):
    with Timer("Part A: logit lens"):
        shots = "".join(f'Español: "{e}" - Italiano: "{i}"\n' for e, i in DETOUR_SHOTS)
        rows = []
        for es, it, en in DETOUR_WORDS:
            ti, te, ts = enc(it)["input_ids"][0], enc(en)["input_ids"][0], enc(es)["input_ids"][0]
            if len({ti, te, ts}) < 3: continue           # first tokens must differ
            rows.append((es, it, en, ti, te))
        P_it, P_en = [], []
        for i in range(0, len(rows), 16):
            chunk = rows[i:i + 16]
            seqs = [BOS + enc(shots + f'Español: "{es}" - Italiano: "')["input_ids"] for es, *_ in chunk]
            L = max(map(len, seqs))
            ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
            for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
            with torch.no_grad():
                out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV), output_hidden_states=True)
                last = am.sum(1) - 1
                for l in range(1, N_LAYERS + 1):
                    h = out.hidden_states[l][torch.arange(len(seqs)), last.to(out.hidden_states[l].device)]
                    if l < N_LAYERS: h = FINAL_NORM(h)      # hidden_states[N] already has the final norm
                    p = torch.softmax(LM_HEAD(h.to(LM_HEAD.weight.device)).float(), -1)
                    ti = torch.tensor([r[3] for r in chunk], device=p.device); te = torch.tensor([r[4] for r in chunk], device=p.device)
                    if l == 1: P_it.append([]); P_en.append([])
                    P_it[-1].append(p.gather(1, ti[:, None])[:, 0].cpu().numpy())
                    P_en[-1].append(p.gather(1, te[:, None])[:, 0].cpu().numpy())
            del out
        P_it = np.concatenate([np.stack(x, 1) for x in P_it], 0)     # [words, layers]
        P_en = np.concatenate([np.stack(x, 1) for x in P_en], 0)
        mi, me = P_it.mean(0), P_en.mean(0)
        cut = int(0.9 * N_LAYERS)
        detour_peak = int(np.argmax(me[:cut]) + 1)
        overtake = next((l + 1 for l in range(N_LAYERS) if mi[l] > me[l] and mi[l] > 0.05), None)
        save_json("A_detour.json", dict(words=[r[:3] for r in rows], P_it=P_it, P_en=P_en,
                                        detour_peak_layer=detour_peak, italian_overtakes_at=overtake))
A = load_json("A_detour.json") if have("A_detour.json") else None
if A:
    P_it, P_en = np.array(A["P_it"]), np.array(A["P_en"]); x = np.arange(1, N_LAYERS + 1) / N_LAYERS
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for P, c, lab in [(P_en, HARD, "English word (never asked for)"), (P_it, EASY, "Italian answer")]:
        ax.plot(x, P.mean(0), color=c, lw=2, label=lab)
        ax.fill_between(x, np.percentile(P, 25, 0), np.percentile(P, 75, 0), color=c, alpha=0.15)
    ax.set_xlabel("depth through the model (0 = input, 1 = output)"); ax.set_ylabel("probability")
    ax.set_title(f"Translating Spanish into Italian, the model's working answer passes through English\n({MODEL_ID.split('/')[-1]}, {len(P_it)} words; band = middle half of words)", loc="left")
    ax.legend(frameon=False); savefig(fig, "A_english_detour")
    print(f"English peaks at layer {A['detour_peak_layer']}/{N_LAYERS}; Italian overtakes at {A['italian_overtakes_at']}; "
          f"peak mean P(English) = {P_en.mean(0).max():.3f}")

## Part B: Round-trip recoverability, with the copying loophole closed
Each sentence gets a ladder of translational practices. For each candidate translation t, the same model back-translates t into the source language, **teacher-forced on the original**. For each source token i:

R_i(t) = log p(s_i | t) − log p(s_i | nothing), and the oracle R_i* is the same with the source text itself as input (copying). A token's **leftover** is ℓ_i = 1 − clip(R_i / R_i*), its weight is w_i = R_i*, and d = Σ w ℓ / Σ w is the share of the source's information the translation fails to carry.

**Fixes since last time:**
- Source words copied into t (ignoring accents) are masked as `[·]` before scoring. Genuine cognate translations are left alone.
- "Keep the word" (B2) is reported separately as *what borrowing buys*.
- **Sense recovery** asks whether the intended sense can be recovered from t, scaled so that 0 = no better than nothing and 1 = as good as the source.

In [ ]:
RUNGS = ["B0", "B1", "B2", "B3", "B5", "B6"]
RUNG_NAME = {"B0": "word for word", "B1": "standard", "B2": "keep the word", "B3": "inline gloss",
             "B5": "footnote", "B6": "thick translation"}
NON_BORROW = ["B0", "B1", "B3", "B5", "B6"]
MAXTOK = {"B0": 64, "B1": 64, "B2": 72, "B3": 120, "B5": 150, "B6": 260}

def rung_user(r, s, L):
    n = LANG_NAME[L]
    return {
        "B0": f"Give a word-for-word English gloss of this {n} sentence: one English word for each {n} word, in the original order. Reply with the gloss only.\n\n{s}",
        "B1": f"Translate this {n} sentence into natural English. Reply with the translation only.\n\n{s}",
        "B2": f"Translate this {n} sentence into English, but keep any word that has no good English equivalent in the original {n}, in italics with *asterisks*. Reply with the translation only.\n\n{s}",
        "B3": f"Translate this {n} sentence into English. After each word or phrase that is hard to translate, add a short gloss in square brackets (at most 8 words). Do not quote the {n} words. Reply with the translation only.\n\n{s}",
        "B5": f"Translate this {n} sentence into English, then add one footnote (at most 40 words) explaining what the translation loses. Do not quote the {n} words. Format: the translation, a blank line, then 'Note: ...'.\n\n{s}",
        "B6": f"Translate this {n} sentence into English, then write one paragraph (at most 120 words) opening up the meanings, connotations and grammar of its hard parts, so that an English reader understands what the original says. Do not quote the {n} words.\n\n{s}",
    }[r]

if RUN_B and not have("B_ladder.json"):
    with Timer("Part B: generating the ladder"):
        ladder = {S["id"]: {} for S in SENTENCES}
        for r in RUNGS:
            outs = generate([chat_prompt(rung_user(r, S["text"], S["lang"])) for S in SENTENCES], max_new_tokens=MAXTOK[r])
            for S, o in zip(SENTENCES, outs): ladder[S["id"]][r] = o
        save_json("B_ladder.json", ladder)
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
if ladder:
    for S in SENTENCES[:3] + [S for S in SENTENCES if S["id"] == "tren"]:
        print(f"\n{S['text']}"); [print(f"  {r:3s} {RUNG_NAME[r]:18s} {ladder[S['id']][r][:150]!r}") for r in RUNGS]

In [ ]:
def mask_source_words(t, s):
    """Replace words in t that copy a source word of >= 4 letters (ignoring accents and a plural s) with '[·]'.
    Proper nouns are left alone. Genuine cognate translations (moment / momento) are NOT masked: they are translations."""
    import unicodedata
    def norm(w): return "".join(c for c in unicodedata.normalize("NFD", w.lower()) if unicodedata.category(c) != "Mn")
    src = set()
    for k, (w, a, b) in enumerate(words_with_spans(s)):
        if len(w) >= 4 and not (w[0].isupper() and k > 0): src |= {norm(w), norm(w) + "s", norm(w).rstrip("s")}
    n = [0]
    def rep(m):
        if norm(m.group(0)) in src: n[0] += 1; return "[·]"
        return m.group(0)
    return WORD_RE.sub(rep, t), n[0]

def bt_user(t, L): return f"Translate this English text into {LANG_NAME[L]}. Reply with the translation only.\n\n{t}"
def copy_user(s, L): return f"Translate this {LANG_NAME[L]} text into {LANG_NAME[L]}. Reply with the translation only.\n\n{s}"
def sense_user(ctx, span, L):
    return (f"Here is a text:\n\n{ctx}\n\nIn the original {LANG_NAME[L]} sentence, what did \"{span}\" mean in this context? "
            f"Answer in one line.")
EMPTY, DELTA = "(no text)", 0.5

def token_word_map(offsets, words):
    return [next((k for k, (_, wa, wb) in enumerate(words) if y > x and y > wa and x < wb), None) for (x, y) in offsets]

def drt_from(lp_t, lp_0, lp_s, offsets, words):
    R, Rs = lp_t - lp_0, lp_s - lp_0
    valid = Rs > DELTA
    w = np.where(valid, Rs, 0.0)
    ell = np.where(valid, 1 - np.clip(R / np.where(valid, Rs, 1.0), 0, 1), 0.0)
    d = float((w * ell).sum() / w.sum()) if w.sum() > 0 else float("nan")
    W, WL = np.zeros(len(words)), np.zeros(len(words))
    for i, k in enumerate(token_word_map(offsets, words)):
        if k is not None: W[k] += w[i]; WL[k] += w[i] * ell[i]
    return d, np.where(W > 0, WL / np.maximum(W, 1e-12), np.nan), W

if RUN_B and ladder and not have("B_roundtrip.json"):
    with Timer("Part B: round-trip scoring"):
        items, meta = [], []
        for S in SENTENCES:
            s, L = S["text"], S["lang"]
            items += [(chat_prompt(bt_user(EMPTY, L)), s), (chat_prompt(copy_user(s, L)), s)]
            meta += [(S["id"], "empty", None), (S["id"], "oracle", None)]
            for r in RUNGS:
                t = ladder[S["id"]][r]; tm, nm = mask_source_words(t, s)
                items += [(chat_prompt(bt_user(tm, L)), s), (chat_prompt(bt_user(t, L)), s)]
                meta += [(S["id"], r, "masked"), (S["id"], r, "raw")]
        sc = score([p for p, _ in items], [c for _, c in items])
        got = {(m[0], m[1], m[2]): r for m, r in zip(meta, sc)}

        # sense recovery
        s_items, s_meta = [], []
        for S in SENTENCES:
            s, L = S["text"], S["lang"]
            for span, gloss in S["hot"].items():
                ctxs = {"source": s, "empty": EMPTY}
                for r in RUNGS:
                    t = ladder[S["id"]][r]
                    ctxs[r] = t if r == "B2" else mask_source_words(t, s)[0]
                for k, ctx in ctxs.items():
                    s_items.append((chat_prompt(sense_user(ctx, span, L)), gloss)); s_meta.append((S["id"], span, k))
        ssc = score([p for p, _ in s_items], [c for _, c in s_items])
        sense_lp = defaultdict(dict)
        for (sid, span, k), r in zip(s_meta, ssc): sense_lp[(sid, span)][k] = float(r["lp"].mean())

        out = {}
        for S in SENTENCES:
            sid, s = S["id"], S["text"]; words = words_with_spans(s)
            lp0, lps = got[(sid, "empty", None)]["lp"], got[(sid, "oracle", None)]["lp"]
            offs = got[(sid, "empty", None)]["offsets"]
            rec = dict(words=[w for w, _, _ in words], rungs={})
            for r in RUNGS:
                t = ladder[sid][r]
                dm, wl_m, W = drt_from(got[(sid, r, "masked")]["lp"], lp0, lps, offs, words)
                du, wl_u, _ = drt_from(got[(sid, r, "raw")]["lp"], lp0, lps, offs, words)
                rec["rungs"][r] = dict(len=ntok(t), d_masked=dm, d_raw=du, word_left_masked=wl_m, word_left_raw=wl_u,
                                       n_masked=mask_source_words(t, s)[1])
            rec["word_weight"] = W
            sr = {}
            for span in S["hot"]:
                v = sense_lp[(sid, span)]; denom = v["source"] - v["empty"]
                sr[span] = dict(raw=v, norm={r: (v[r] - v["empty"]) / denom if abs(denom) > 1e-6 else float("nan") for r in RUNGS})
            rec["sense"] = sr
            out[sid] = rec
        save_json("B_roundtrip.json", out)

In [ ]:
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
if B:
    ctrl = [B[S["id"]]["rungs"]["B1"]["d_masked"] for S in SENTENCES if S["cat"] == "control"]
    EPS = float(np.nanpercentile(ctrl, 90))
    def curve_stats(rec, rungs):
        L0 = rec["rungs"]["B1"]["len"]
        cands = [(rec["rungs"][r]["len"], rec["rungs"][r]["d_raw" if r == "B2" else "d_masked"]) for r in rungs]
        ok = [l for l, d in cands if d <= EPS]
        U = math.log2(max(min(ok), 1) / L0) if ok else math.log2(max(l for l, _ in cands) / L0)
        budgets = L0 * 2 ** np.linspace(0, 4, 9)
        Dl = [min([d for l, d in cands if l <= bgt] or [np.nan]) for bgt in budgets]
        return dict(U=max(U, 0.0), censored=not ok, A=float(np.nanmean(Dl)))
    summary = {}
    for sid, rec in B.items():
        summary[sid] = dict(cat=SID[sid]["cat"], lang=SID[sid]["lang"], **curve_stats(rec, NON_BORROW),
                            U_with_borrowing=curve_stats(rec, NON_BORROW + ["B2"])["U"],
                            d_B1=rec["rungs"]["B1"]["d_masked"],
                            borrow_gain=rec["rungs"]["B1"]["d_masked"] - rec["rungs"]["B2"]["d_raw"],
                            borrow_gain_masked=rec["rungs"]["B1"]["d_masked"] - rec["rungs"]["B2"]["d_masked"])
    # hot words vs other content words (standard translation, masked)
    hot_v, oth_v = [], []
    for S in SENTENCES:
        rec = B[S["id"]]; wl = np.array(rec["rungs"]["B1"]["word_left_masked"], float); W = np.array(rec["word_weight"])
        hot_idx = set()
        for span in S["hot"]:
            sp = find_phrase(S["text"], span)
            if sp: hot_idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
        for k, w in enumerate(rec["words"]):
            if W[k] <= 0 or np.isnan(wl[k]): continue
            if k in hot_idx: hot_v.append(wl[k])
            elif len(w) >= 4: oth_v.append(wl[k])
    B_auroc = auroc(hot_v, oth_v)
    print(f"tolerance ε (90th pct of control d at 'standard') = {EPS:.3f}")
    print(f"hot words vs other content words, leftover AUROC = {B_auroc:.3f} ({len(hot_v)} hot, {len(oth_v)} other)\n")
    print(f"{'category':15s} {'n':>2s} {'U (doublings)':>13s} {'cens':>4s} {'A':>6s} {'d std':>6s} {'borrow buys':>11s} {'…masked':>8s}")
    cats = ["untranslatable", "construction", "idiom", "control"]
    for c in cats:
        v = [x for x in summary.values() if x["cat"] == c]
        print(f"{c:15s} {len(v):2d} {np.mean([x['U'] for x in v]):13.2f} {sum(x['censored'] for x in v):4d} "
              f"{np.mean([x['A'] for x in v]):6.3f} {np.nanmean([x['d_B1'] for x in v]):6.3f} "
              f"{np.nanmean([x['borrow_gain'] for x in v]):11.3f} {np.nanmean([x['borrow_gain_masked'] for x in v]):8.3f}")
    fa = B["duende"]; wl = dict(zip(fa["words"], fa["rungs"]["B1"]["word_left_masked"]))
    print(f"\nMe gusta mucho el duende (standard translation: {ladder['duende']['B1']!r})")
    print("  leftover: " + ", ".join(f"{w} {wl.get(w, float('nan')):.2f}" for w in ["duende", "gusta", "mucho"])
          + "   [Llama-3.1-8B last time: duende 0.50, gusta 0.23, mucho 0.01]")
    print("  sense recovered (0 = none, 1 = as good as the source):")
    for span, v in fa["sense"].items():
        print(f"    {span:8s} " + "  ".join(f"{RUNG_NAME[r]} {v['norm'][r]:.2f}" for r in RUNGS))
    save_json("B_summary.json", dict(epsilon=EPS, auroc_hot_words=B_auroc, per_sentence=summary))

In [ ]:
if B:
    def heat_row(ax, y, words, vals, title, sub):
        """Words set in a row, each shaded by its leftover; positions measured from the rendered text."""
        fig_ = ax.figure; r = fig_.canvas.get_renderer(); inv = ax.transData.inverted(); x = 0.0
        ax.text(0, y + 0.45, title, fontsize=9, color="#666", ha="left")
        for w, v in zip(words, vals):
            bad = v is None or np.isnan(v)
            c = GREY if bad else plt.cm.Oranges(0.08 + 0.85 * float(np.clip(v, 0, 1)))
            t = ax.text(x, y, w, fontsize=17, style="italic", family="serif", va="center", ha="left",
                        bbox=dict(boxstyle="round,pad=0.25", fc=c, ec="none"))
            bb = t.get_window_extent(renderer=r); (x0, _), (x1, _) = inv.transform([[bb.x0, bb.y0], [bb.x1, bb.y1]])
            ax.text((x0 + x1) / 2, y - 0.36, "" if bad else f"{v:.2f}", fontsize=7.5, color="#555", ha="center")
            x = x1 + 0.03
        ax.text(0, y - 0.66, sub, fontsize=9, color=INK, ha="left")
    show = [s for s in ["duende", "olvido", "pelo", "estrenar", "tren"] if s in B]
    fig, ax = plt.subplots(figsize=(9, 1.45 * len(show))); ax.axis("off")
    ax.set_xlim(-0.02, 1.25); ax.set_ylim(-1.5 * len(show) + 0.6, 0.9)
    for k, sid in enumerate(show):
        rec = B[sid]; S = SID[sid]
        wds = [w for w, _, _ in words_with_spans(S["text"])]
        heat_row(ax, -1.5 * k, wds, rec["rungs"]["B1"]["word_left_masked"], S["cat"], "→ " + ladder[sid]["B1"][:110])
    ax.set_xlim(-0.02, 1.25); ax.set_ylim(-1.5 * len(show) + 0.6, 0.9)
    ax.set_title("Where each sentence resists translation: darker words are harder to recover from the English\n"
                 f"(share of each word's information lost after a standard translation; {MODEL_ID.split('/')[-1]})", loc="left", fontsize=10)
    savefig(fig, "B1_heatmaps")

    show = [s for s in ["duende", "sobremesa", "pelo", "tren"] if s in B]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for sid in show:
        rec = B[sid]; col = EASY if SID[sid]["cat"] == "control" else HARD
        pts = sorted((rec["rungs"][r]["len"], rec["rungs"][r]["d_masked"], r) for r in NON_BORROW)
        env, best = [], 1.0
        for l, d, r in pts: best = min(best, d); env.append((l, best))
        ax.step([l for l, _ in env], [d for _, d in env], where="post", color=col, lw=2, alpha=0.9)
        ax.scatter([l for l, _, _ in pts], [d for _, d, _ in pts], color=col, s=18, zorder=3)
        ax.text(env[-1][0] * 1.05, env[-1][1], SID[sid]["text"][:28] + "…", fontsize=8, color=col, va="center")
    ax.axhline(EPS, color="#888", ls="--", lw=1); ax.text(ax.get_xlim()[0], EPS, "  gets across", fontsize=8, color="#888", va="bottom")
    ax.set_xscale("log"); ax.set_xlabel("length of the English (tokens, log scale): word for word → standard → gloss → footnote → thick")
    ax.set_ylabel("share of meaning not carried"); ax.set_title("How much English does it take?", loc="left")
    savefig(fig, "B2_distortion_vs_length")

    spans = [(sid, sp) for sid in B for sp in B[sid]["sense"]][:12]
    if spans:
        fig, ax = plt.subplots(figsize=(9, 3.6)); w = 0.15
        rr = ["B1", "B2", "B3", "B5", "B6"]; cols = [GREY, LOAN, "#e3a587", "#d07a55", HARD]
        for k, r in enumerate(rr):
            ax.bar(np.arange(len(spans)) + (k - 2) * w, np.clip([B[s]["sense"][sp]["norm"][r] for s, sp in spans], -0.5, 1.5), w, color=cols[k], label=RUNG_NAME[r])
        ax.set_xticks(np.arange(len(spans))); ax.set_xticklabels([sp for _, sp in spans], rotation=30, ha="right", style="italic")
        ax.axhline(1, color="#888", lw=0.8, ls=":"); ax.axhline(0, color="#888", lw=0.8)
        ax.set_ylabel("intended sense recovered\n(0 = none, 1 = as from the original)"); ax.set_ylim(-0.55, 1.55)
        ax.set_title("Does the English carry the sense the sentence means?", loc="left"); ax.legend(frameon=False, fontsize=8, ncol=5)
        savefig(fig, "B3_sense_recovery")

## Part C: Round-trip spectrum (canonical correlations)

**Setup.**
- Take aligned word pairs (x_i, y_i) from a parallel corpus. Each vector is the word's residual-stream state at layer ℓ, in its own sentence, fed raw.
- Centre both languages and take the ridge covariances Σxx, Σyy, Σxy.
- The least-squares maps are A = Σyx Σxx⁻¹ and B = Σxy Σyy⁻¹.

**The identity.** The round trip M = BA is similar to KKᵀ, where K = Σxx^{-½} Σxy Σyy^{-½}. So **its eigenvalues are the squared canonical correlations ρ_k²**.

**Per word.**
- Whiten x and take its canonical coordinates c_k.
- Retention after n round trips: R_n = Σ ρ_k^{2n} c_k² / Σ c_k².
- Untranslatability U = 1 − R₁, and the half-life is the n at which R_n = ½.
- Raw-metric check: U_raw = ‖x − Mx‖² / ‖x‖².

**English bias.** The held-out share of each language's variance that is predictable from the other: E_es (Spanish from English) vs E_en (English from Spanish).

**Honesty rules.**
- All ρ are **held-out**: fitted on one set of sentences, measured on different ones.
- A shuffled-pairs null sets the noise floor.
- The ridge strength is chosen on one half of the held-out set and reported on the other half.

In [ ]:
def pick_layers():
    f = [0.25, 0.4, 0.5, 0.6, 0.75]
    ls = {max(1, min(N_LAYERS - 1, round(x * N_LAYERS))) for x in f}
    if A and A.get("detour_peak_layer"): ls.add(int(min(A["detour_peak_layer"], N_LAYERS - 1)))
    return sorted(ls)
C_LAYERS = pick_layers(); print("Part C layers:", C_LAYERS)

def load_pairs(lang, n):
    """n parallel sentence pairs (x, en, source); a mix of domains so 'conventional practice' isn't one register."""
    from datasets import load_dataset
    # Europarl first: if it fails to load, OPUS-100 takes the whole quota
    sources = ([("Helsinki-NLP/europarl", "en-es")] if lang == "es" else []) + [("Helsinki-NLP/opus-100", f"en-{lang}")]
    want, pairs, seen = int(n * 1.3), [], set()
    for k, (name, cfg) in enumerate(sources):
        quota = (want - len(pairs)) // (len(sources) - k)
        try: ds = load_dataset(name, cfg, split="train", streaming=True)
        except Exception as e: print(f"  could not load {name}/{cfg}: {e}"); continue
        got = 0
        for ex in ds:
            tr = ex.get("translation", {}); en, xx = tr.get("en"), tr.get(lang)
            if not en or not xx: continue
            ne, nx = len(en.split()), len(xx.split())
            if not (8 <= ne <= 40 and 8 <= nx <= 40) or abs(ne - nx) > 0.5 * max(ne, nx): continue
            if sum(ch.isdigit() for ch in xx) > 6 or xx in seen: continue
            seen.add(xx); pairs.append((xx.strip(), en.strip(), name)); got += 1
            if got >= quota: break
        print(f"  {name}/{cfg}: {got} pairs")
    random.Random(SEED).shuffle(pairs)
    return pairs[:n]

def words_for(lang, texts):
    """Word tokens (text, start, end, morph, pos). Spanish uses spaCy when available, for morphology."""
    if lang == "es":
        try:
            import spacy
            nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
            out = []
            for doc in nlp.pipe(texts, batch_size=256):
                out.append([(t.text, t.idx, t.idx + len(t.text), str(t.morph), t.pos_) for t in doc
                            if WORD_RE.fullmatch(t.text)])
            return out
        except Exception as e:
            print("  spaCy unavailable, using regex words (no morphology probes):", e)
    return [[(w, a, b, "", "") for w, a, b in words_with_spans(t)] for t in texts]

_AL = None
def align_pairs(src_words, tgt_words, batch=64, layer=8):
    """1-to-1 word links (i, j) per sentence pair: mutual nearest neighbours of XLM-R (layer 8) word embeddings,
    i.e. SimAlign's high-precision 'argmax' method, implemented here to avoid a fragile dependency."""
    global _AL
    from transformers import AutoTokenizer, AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if _AL is None:
        _AL = (AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base"),
               AutoModel.from_pretrained("FacebookAI/xlm-roberta-base").to(dev).eval())
    atok, amod = _AL
    def embed(word_lists):
        wl_ = [[w[0] for w in wl] or ["."] for wl in word_lists]
        e = atok(wl_, is_split_into_words=True, return_tensors="pt", padding=True, truncation=True, max_length=256)
        with torch.no_grad():
            H = amod(**{k: v.to(dev) for k, v in e.items()}, output_hidden_states=True).hidden_states[layer]
        out = []
        for b, wl in enumerate(wl_):
            M = torch.zeros(len(wl), H.shape[1], device=dev)
            for p, w in enumerate(e.word_ids(b)):
                if w is not None: M[w, p] = 1.0
            cnt = M.sum(1, keepdim=True)
            out.append((torch.nn.functional.normalize((M @ H[b].float()) / cnt.clamp_min(1), dim=-1), cnt[:, 0] > 0))
        return out
    links = []
    for i in range(0, len(src_words), batch):
        es, et = embed(src_words[i:i + batch]), embed(tgt_words[i:i + batch])
        for (a, va), (b, vb), sw, tw in zip(es, et, src_words[i:i + batch], tgt_words[i:i + batch]):
            if not sw or not tw: links.append([]); continue
            S = a @ b.T; S[~va] = -2; S[:, ~vb] = -2
            fwd, bwd = S.argmax(1), S.argmax(0)
            links.append([(k, int(fwd[k])) for k in range(len(sw)) if int(bwd[fwd[k]]) == k and float(S[k, fwd[k]]) > -1
                          and not any(ch.isdigit() for ch in sw[k][0] + tw[int(fwd[k])][0])])
    return links

In [ ]:
class CovAcc:
    def __init__(self, d, dev):
        z = lambda *s: torch.zeros(*s, dtype=torch.float64, device=dev)
        self.n, self.sx, self.sy = 0, z(d), z(d)
        self.Sxx, self.Syy, self.Sxy = z(d, d), z(d, d), z(d, d)
    def add(self, X, Y):
        X, Y = X.double(), Y.double()
        self.n += X.shape[0]; self.sx += X.sum(0); self.sy += Y.sum(0)
        self.Sxx += X.T @ X; self.Syy += Y.T @ Y; self.Sxy += X.T @ Y
    def finish(self):
        n = self.n; mx, my = self.sx / n, self.sy / n
        return dict(n=n, mx=mx.cpu(), my=my.cpu(),
                    Cxx=((self.Sxx - n * torch.outer(mx, mx)) / (n - 1)).cpu(),
                    Cyy=((self.Syy - n * torch.outer(my, my)) / (n - 1)).cpu(),
                    Cxy=((self.Sxy - n * torch.outer(mx, my)) / (n - 1)).cpu())

def word_states(texts, word_lists, layers, batch=32):
    """For each text: dict layer -> [n_words, d] float32 (word = mean of its tokens), plus a validity mask
    (words containing the first content token, an attention sink, are invalid)."""
    out = []
    for i in range(0, len(texts), batch):
        enc_ = [encode_raw(t) for t in texts[i:i + batch]]
        L = max(len(e[0]) for e in enc_)
        ids = torch.full((len(enc_), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, (s, _) in enumerate(enc_): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        st = capture(ids.to(INPUT_DEV), am.to(INPUT_DEV), layers)
        for b, (s, offs) in enumerate(enc_):
            wl = word_lists[i + b]; first = len(BOS)
            toks = [tokens_in_span(offs, w[1], w[2]) for w in wl]
            valid = np.array([len(t) > 0 and first not in t for t in toks], bool)
            M = torch.zeros(len(wl), len(s))
            for k, t in enumerate(toks):
                if t: M[k, t] = 1.0 / len(t)
            M = M.to(st[layers[0]].device)
            out.append(({l: M @ st[l][b, :len(s)].float() for l in layers}, valid))
        del st
    return out

def collect_pair(lang):
    tag = f"C_{lang}"
    cov_paths = {l: os.path.join(CACHE, f"{tag}_cov_L{l}.pt") for l in C_LAYERS}
    held_path = os.path.join(CACHE, f"{tag}_held.pt")
    if all(os.path.exists(p) for p in cov_paths.values()) and os.path.exists(held_path): return
    with Timer(f"Part C [{lang}]: corpus"):
        pairs = load_pairs(lang, N_SENT_FIT + N_SENT_HELD)
        xs, ens = [p[0] for p in pairs], [p[1] for p in pairs]
        wx, we = words_for(lang, xs), words_for("en", ens)
    with Timer(f"Part C [{lang}]: word alignment ({len(pairs)} pairs)"):
        links = align_pairs(wx, we)
    n_fit = len(pairs) - N_SENT_HELD
    acc = {l: CovAcc(D_MODEL, INPUT_DEV) for l in C_LAYERS}
    held = {l: ([], []) for l in C_LAYERS}; held_meta = []
    with Timer(f"Part C [{lang}]: collecting states at layers {C_LAYERS}"):
        B_ = 32
        for i in range(0, len(pairs), B_):
            sx = word_states(xs[i:i + B_], wx[i:i + B_], C_LAYERS)
            sy = word_states(ens[i:i + B_], we[i:i + B_], C_LAYERS)
            for b in range(len(sx)):
                k = i + b; (rx, vx), (ry, vy) = sx[b], sy[b]
                lk = [(a, c) for a, c in links[k] if vx[a] and vy[c]]
                if not lk: continue
                ia = torch.tensor([a for a, _ in lk]); ic = torch.tensor([c for _, c in lk])
                for l in C_LAYERS:
                    X, Y = rx[l][ia.to(rx[l].device)], ry[l][ic.to(ry[l].device)]
                    if k < n_fit: acc[l].add(X, Y)
                    else: held[l][0].append(X.half().cpu()); held[l][1].append(Y.half().cpu())
                if k >= n_fit:
                    held_meta += [dict(sent=k, x=wx[k][a][0], y=we[k][c][0], morph=wx[k][a][3], pos=wx[k][a][4]) for a, c in lk]
            if (i // B_) % 100 == 0: print(f"    {i}/{len(pairs)} sentences, {acc[C_LAYERS[0]].n} fit pairs")
    for l in C_LAYERS: torch.save(acc[l].finish(), cov_paths[l])
    torch.save(dict(X={l: torch.cat(held[l][0]) for l in C_LAYERS}, Y={l: torch.cat(held[l][1]) for l in C_LAYERS},
                    meta=held_meta, n_fit_pairs=acc[C_LAYERS[0]].n, n_sent=len(pairs)), held_path)
    del acc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

if RUN_C:
    for lang in PAIRS: collect_pair(lang)

In [ ]:
CDEV = "cuda" if torch.cuda.is_available() else "cpu"

def inv_sqrt(C):
    e, V = torch.linalg.eigh(C); return (V * e.clamp_min(1e-12).rsqrt()) @ V.T

def cca_fit(cov, lam):
    Cxx, Cyy, Cxy = (cov[k].to(CDEV) for k in ["Cxx", "Cyy", "Cxy"]); d = Cxx.shape[0]
    I = torch.eye(d, dtype=Cxx.dtype, device=CDEV)
    Cxx_r = Cxx + lam * torch.trace(Cxx) / d * I; Cyy_r = Cyy + lam * torch.trace(Cyy) / d * I
    Wx, Wy = inv_sqrt(Cxx_r), inv_sqrt(Cyy_r)
    U, S, Vh = torch.linalg.svd(Wx @ Cxy @ Wy)
    return dict(lam=lam, rho_in=S, U=U, Wx=Wx, Wy=Wy, A=Wx @ U, B=Wy @ Vh.T, Cxx_r=Cxx_r, Cyy_r=Cyy_r,
                Cxx=Cxx, Cyy=Cyy, Cxy=Cxy, mx=cov["mx"].to(CDEV), my=cov["my"].to(CDEV))

def colcorr(P, Q):
    P = P - P.mean(0); Q = Q - Q.mean(0)
    return (P * Q).sum(0) / (P.norm(dim=0) * Q.norm(dim=0) + 1e-12)

def heldout_rho(fit, X, Y):
    return colcorr((X - fit["mx"]) @ fit["A"], (Y - fit["my"]) @ fit["B"])

def r2(Yc, Yp):
    raw = 1 - ((Yc - Yp) ** 2).sum() / (Yc ** 2).sum()
    per = 1 - ((Yc - Yp) ** 2).sum(0) / ((Yc - Yc.mean(0)) ** 2).sum(0).clamp_min(1e-12)
    return float(raw), float(per.mean())

def analyse_layer(lang, l):
    cov = torch.load(os.path.join(CACHE, f"C_{lang}_cov_L{l}.pt"), weights_only=False)
    held = torch.load(os.path.join(CACHE, f"C_{lang}_held.pt"), weights_only=False)
    X, Y = held["X"][l].to(CDEV).double(), held["Y"][l].to(CDEV).double()
    sent = torch.tensor([m["sent"] for m in held["meta"]])
    half = (sent % 2 == 0).to(CDEV)                       # split held-out by sentence
    best = None
    for lam in LAMBDAS:
        fit = cca_fit(cov, lam); rho = heldout_rho(fit, X[half], Y[half])
        s50 = float(rho[:50].sum())
        if best is None or s50 > best[0]: best = (s50, lam)
        del fit
    fit = cca_fit(cov, best[1])
    rho_h = heldout_rho(fit, X[~half], Y[~half])          # reported on the other half
    perm = torch.randperm(int((~half).sum()), generator=torch.Generator().manual_seed(SEED)).to(CDEV)
    rho_null = heldout_rho(fit, X[~half], Y[~half][perm])
    thr = float(torch.quantile(rho_null.abs().float(), 0.99))
    shared = torch.nonzero(rho_h > thr)[:, 0]
    # English bias: held-out share of each language's variance predictable from the other
    Xc, Yc = X[~half] - fit["mx"], Y[~half] - fit["my"]
    Gxy = torch.linalg.solve(fit["Cxx_r"], fit["Cxy"]); Gyx = torch.linalg.solve(fit["Cyy_r"], fit["Cxy"].T)
    E_en, E_en_std = r2(Yc, Xc @ Gxy)          # English predicted from the other language
    E_x, E_x_std = r2(Xc, Yc @ Gyx)            # other language predicted from English
    spec_x = float(torch.trace(fit["Cxy"] @ Gyx) / torch.trace(fit["Cxx"]))
    res = dict(layer=l, lam=best[1], n_fit_pairs=int(cov["n"]), n_held_pairs=int((~half).sum()),
               rho_in=fit["rho_in"][:1024].cpu().numpy(), rho_held=rho_h[:1024].cpu().numpy(),
               null_thr=thr, n_shared=int(len(shared)), top50_held=float(rho_h[:50].sum()), top50_in=float(fit["rho_in"][:50].sum()),
               E_en_from_x=E_en, E_x_from_en=E_x, E_en_from_x_std=E_en_std, E_x_from_en_std=E_x_std,
               bias=E_en - E_x, bias_std=E_en_std - E_x_std, E_x_spectral_in_sample=spec_x)
    return res, fit, rho_h, shared, Gxy, Gyx

C_RES, C_FITS, BEST_LAYER = {}, {}, {}
if RUN_C:
    for lang in PAIRS:
        with Timer(f"Part C [{lang}]: CCA at {len(C_LAYERS)} layers"):
            rows = []
            for l in C_LAYERS:
                out = analyse_layer(lang, l); res = out[0]; rows.append(res); del out
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                print(f"  L{l:3d}  λ={res['lam']:.0e}  shared dirs={res['n_shared']:5d}  top-50 ρ held/in-sample="
                      f"{res['top50_held']:.1f}/{res['top50_in']:.1f}  E_en←{lang}={res['E_en_from_x']:.3f}  "
                      f"E_{lang}←en={res['E_x_from_en']:.3f}  bias={res['bias']:+.3f}")
            save_json(f"C_spectrum_{lang}.json", rows); C_RES[lang] = rows
            BEST_LAYER[lang] = max(rows, key=lambda r: r["top50_held"])["layer"]
    print("best (most shared) layer:", BEST_LAYER)
    # refit and keep only what later cells need: the best layer per language, plus the detour layer for Spanish (Part D)
    keep = {(lang, BEST_LAYER[lang]) for lang in C_RES}
    if A and "es" in C_RES: keep.add(("es", min(C_LAYERS, key=lambda l: abs(l - A["detour_peak_layer"]))))
    for (lang, l) in sorted(keep):
        res, fit, rho_h, shared, Gxy, Gyx = analyse_layer(lang, l)
        C_FITS[(lang, l)] = dict(fit={k: fit[k] for k in ["Wx", "U", "A", "Cxx_r", "mx"]}, rho_h=rho_h, shared=shared, Gxy=Gxy, Gyx=Gyx)
        del fit; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
def span_scores(lang, l, sentences):
    """U (whitened), U_raw, half-life and #tokens for every word of the given test sentences."""
    F_ = C_FITS[(lang, l)]; fit = F_["fit"]; rho2 = F_["rho_h"].clamp(0, 1).double() ** 2
    texts = [S["text"] for S in sentences]
    wl = [[(w, a, b) for w, a, b in words_with_spans(t)] for t in texts]
    st = word_states(texts, wl, [l])
    ns = np.linspace(0, 50, 501); out = {}
    for S, words, (rec, valid) in zip(sentences, wl, st):
        X = rec[l].double().to(CDEV) - fit["mx"]
        c2 = ((X @ fit["Wx"]) @ fit["U"]) ** 2               # canonical coordinates squared
        Rn = torch.stack([(c2 * rho2 ** n).sum(1) / c2.sum(1) for n in ns], 1).cpu().numpy()
        Xrt = (X @ F_["Gxy"]) @ F_["Gyx"]
        U_raw = (((X - Xrt) ** 2).sum(1) / (X ** 2).sum(1)).cpu().numpy()
        half = [float(ns[np.argmax(r <= 0.5)]) if (r <= 0.5).any() else 50.0 for r in Rn]
        out[S["id"]] = dict(words=[w for w, _, _ in words], U=(1 - Rn[:, 10]).tolist(), U_raw=U_raw.tolist(),
                            half_life=half, R_curve=Rn[:, ::10][:, :11].tolist(),
                            ntok=[len(enc(w)["input_ids"]) for w, _, _ in words], valid=valid.tolist())
    return out

def hot_word_index(S):
    idx = set()
    for span in S["hot"]:
        sp = find_phrase(S["text"], span)
        if sp: idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
    return idx

C_SPANS = {}
if RUN_C and C_RES:
    for lang in C_RES:
        test = [S for S in SENTENCES if S["lang"] == lang]
        if not test: continue
        rows = {}
        for (lg, l) in C_FITS:
            if lg != lang: continue
            sc = span_scores(lang, l, test)
            hot, oth = defaultdict(list), defaultdict(list)
            for S in test:
                r = sc[S["id"]]; hi = hot_word_index(S)
                for k, w in enumerate(r["words"]):
                    if not r["valid"][k]: continue
                    tgt = hot if k in hi else (oth if len(w) >= 4 else None)
                    if tgt is None: continue
                    for m in ["U", "U_raw", "half_life", "ntok"]: tgt[m].append(r[m][k])
            au = {m: auroc(hot[m], oth[m]) for m in ["U", "U_raw", "half_life", "ntok"]}
            # agreement with Part B (sentence-level, content words weighted by round-trip information weight)
            agree = None
            if B:
                ui, ub = [], []
                for S in test:
                    r = sc[S["id"]]; W = np.array(B[S["id"]]["word_weight"]); u = np.array(r["U"]); v = np.array(r["valid"])
                    if (W[v] > 0).any(): ui.append(float((W[v] * u[v]).sum() / W[v].sum())); ub.append(summary[S["id"]]["A"])
                agree = spearman(ui, ub)
            rows[l] = dict(scores=sc, auroc=au, spearman_with_roundtrip_A=agree, n_hot=len(hot["U"]), n_other=len(oth["U"]))
            print(f"[{lang}] layer {l}: AUROC hot vs other content words — U {au['U']:.2f}, U_raw {au['U_raw']:.2f}, "
                  f"half-life {1-au['half_life']:.2f} (shorter = harder), #tokens baseline {au['ntok']:.2f}; "
                  f"agreement with round trip ρ_s={agree if agree is None else round(agree, 2)}")
            if lang == "es" and "duende" in sc:
                r = sc["duende"]; d = dict(zip(r["words"], zip(r["U"], r["half_life"])))
                print("   Me gusta mucho el duende: " + ", ".join(f"{w} U={d[w][0]:.3f} (half-life {d[w][1]:.1f})" for w in ["duende", "gusta", "mucho"] if w in d))
        C_SPANS[lang] = rows
        save_json(f"C_spans_{lang}.json", rows)

In [ ]:
if RUN_C and "es" in C_RES and C_SPANS.get("es"):
    held = torch.load(os.path.join(CACHE, "C_es_held.pt"), weights_only=False); meta = held["meta"]
    l = BEST_LAYER["es"]; F_ = C_FITS[("es", l)]; fit = F_["fit"]
    if any(m["morph"] for m in meta):
        from sklearn.linear_model import RidgeClassifier
        from sklearn.model_selection import cross_val_score
        Z = (((held["X"][l].to(CDEV).double() - fit["mx"]) @ fit["Wx"]) @ fit["U"]).float().cpu().numpy()
        Z /= Z.std(0, keepdims=True) + 1e-8
        sh = np.zeros(Z.shape[1], bool); sh[F_["shared"].cpu().numpy()] = True
        def feat(m, key): return dict(kv.split("=") for kv in m["morph"].split("|") if "=" in kv).get(key)
        tasks = {
            "grammatical gender": [(i, feat(m, "Gender")) for i, m in enumerate(meta) if m["pos"] in ("NOUN", "ADJ") and feat(m, "Gender") in ("Masc", "Fem")],
            "number": [(i, feat(m, "Number")) for i, m in enumerate(meta) if m["pos"] in ("NOUN", "ADJ", "VERB") and feat(m, "Number") in ("Sing", "Plur")],
            "subjunctive vs indicative": [(i, feat(m, "Mood")) for i, m in enumerate(meta) if m["pos"] in ("VERB", "AUX") and feat(m, "Mood") in ("Sub", "Ind")],
            "imperfect vs preterite": [(i, feat(m, "Tense")) for i, m in enumerate(meta) if m["pos"] in ("VERB", "AUX") and feat(m, "Mood") == "Ind" and feat(m, "Tense") in ("Imp", "Past")],
            "noun vs verb (sanity)": [(i, m["pos"]) for i, m in enumerate(meta) if m["pos"] in ("NOUN", "VERB")],
        }
        probe = {}
        for name, items in tasks.items():
            labs = [b for _, b in items]
            if len(items) < 60 or min(labs.count(v) for v in set(labs)) < 20: print(f"  {name}: too few examples"); continue
            ii = np.array([i for i, _ in items]); y = np.array([b == sorted(set(labs))[0] for b in labs])
            clf = RidgeClassifier(alpha=float(len(ii)), class_weight="balanced")
            acc = {k: float(cross_val_score(clf, Z[ii][:, cols], y, cv=5, scoring="balanced_accuracy").mean())
                   for k, cols in [("all", np.ones_like(sh)), ("shared only", sh), ("private only", ~sh)] if cols.any()}
            clf.fit(Z[ii], y); wgt = clf.coef_[0] ** 2
            probe[name] = dict(n=len(ii), shared_share=float(wgt[sh].sum() / wgt.sum()), expected=float(sh.mean()), acc=acc)
            print(f"  {name:28s} n={len(ii):5d}  weight on shared directions {probe[name]['shared_share']:.2f} "
                  f"(chance {sh.mean():.2f})  accuracy all/shared/private = " + "/".join(f"{v:.2f}" for v in acc.values()))
        save_json("C_probes_es.json", dict(layer=l, probes=probe))

In [ ]:
if RUN_C and C_RES:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    ax = axes[0]
    for lang, col in zip(C_RES, [HARD, LOAN, EASY]):
        r = next(x for x in C_RES[lang] if x["layer"] == BEST_LAYER[lang]); k = np.arange(1, len(r["rho_in"]) + 1)
        ax.plot(k, r["rho_in"], color=col, lw=1, alpha=0.5, ls="--")
        ax.plot(k, r["rho_held"], color=col, lw=1.6, label=f"{lang}–en, layer {r['layer']} (held-out)")
        ax.axhline(r["null_thr"], color=col, lw=0.8, ls=":")
    ax.set_xscale("log"); ax.set_xlabel("canonical direction k"); ax.set_ylabel("ρ_k")
    ax.set_title("Round-trip spectrum (dashed: in-sample; dotted: shuffled-pair null)", loc="left", fontsize=9); ax.legend(frameon=False, fontsize=8)
    ax = axes[1]
    for lang, col in zip(C_RES, [HARD, LOAN, EASY]):
        ax.plot([r["layer"] / N_LAYERS for r in C_RES[lang]], [r["top50_held"] / 50 for r in C_RES[lang]], "o-", color=col, label=f"{lang}–en")
    if A: ax.axvline(A["detour_peak_layer"] / N_LAYERS, color="#888", ls="--", lw=1); ax.text(A["detour_peak_layer"] / N_LAYERS, ax.get_ylim()[0], " English detour peak", fontsize=8, color="#888")
    ax.set_xlabel("depth"); ax.set_ylabel("mean held-out ρ, top 50 directions"); ax.legend(frameon=False)
    ax.set_title("Where the two languages share most", loc="left", fontsize=9)
    savefig(fig, "C1_spectrum")

    if C_SPANS.get("es"):
        sc = C_SPANS["es"][BEST_LAYER["es"]]["scores"]
        pick = [("duende", "duende"), ("duende", "gusta"), ("sobremesa", "sobremesa"), ("madrugada", "madrugada"),
                ("tren", "tren"), ("hermano", "hermano"), ("gatos", "gatos")]
        fig, ax = plt.subplots(figsize=(7, 3.6))
        for sid, w in pick:
            if sid not in sc or w not in sc[sid]["words"]: continue
            k = sc[sid]["words"].index(w); col = EASY if SID[sid]["cat"] == "control" else HARD
            ax.plot(range(11), sc[sid]["R_curve"][k], color=col, lw=2, alpha=0.85)
            ax.text(10.2, sc[sid]["R_curve"][k][-1], f"{w} (half-life {sc[sid]['half_life'][k]:.1f})", fontsize=8, style="italic", color=col, va="center")
        ax.axhline(0.5, color="#888", ls=":", lw=1)
        ax.set_xlabel("number of round trips Spanish → English → Spanish (inside the model)"); ax.set_ylabel("share of the word that survives")
        ax.set_title("What survives repeated round trips", loc="left"); ax.set_xlim(0, 13.5)
        savefig(fig, "C2_retention")

    fig, ax = plt.subplots(figsize=(5.5, 3.4)); xs = np.arange(len(C_RES)); w = 0.35
    r_ = [next(x for x in C_RES[lang] if x["layer"] == BEST_LAYER[lang]) for lang in C_RES]
    ax.bar(xs - w / 2, [r["E_en_from_x"] for r in r_], w, color=GREY, label="English, predicted from the other language")
    ax.bar(xs + w / 2, [r["E_x_from_en"] for r in r_], w, color=HARD, label="other language, predicted from English")
    ax.set_xticks(xs); ax.set_xticklabels([f"{LANG_NAME[l]}–English" for l in C_RES]); ax.set_ylabel("share of variation predictable")
    ax.set_title("How much of each language survives the crossing", loc="left"); ax.legend(frameon=False, fontsize=8)
    savefig(fig, "C3_english_bias")

## Part D: Causal mediation. Shared vs private parts of a word
The shared subspace consists of the canonical directions whose held-out ρ beats the null, at the most-shared layer. Each Spanish token state splits obliquely into a shared part and a private part: x_sh = Σ_k (a_kᵀx) Σxx a_k.

1. **Ablation.** For hot words and control words, set the private part (or the shared part, or everything) to its corpus mean. Then measure how much that hurts (a) the model's own **translation** of the sentence and (b) its own **explanation of what the word means** in that sentence.
   - *Understood but not translated* = the private part matters for (b) but not for (a).
2. **Interchange.** Same word, two senses (*el cura* / *la cura*, *el cometa* / *la cometa*, *es* / *está aburrido*, *duende* in a fairy tale vs in flamenco). Patch the word's private part (or shared part, or whole state) from one context into the other, and measure how far the English moves towards the other sense: **flip fraction**, 0 = no change, 1 = fully switched.

Note: the basis is fitted on words in raw sentences and applied to tokens inside a translation prompt, which is an approximation.

In [ ]:
def decomposer(lang, l):
    F_ = C_FITS[(lang, l)]; fit = F_["fit"]
    idx = F_["shared"]
    A_r = fit["A"][:, idx].float(); D_r = (fit["Cxx_r"] @ fit["A"][:, idx]).float(); mx = fit["mx"].float()
    def split(h):
        xc = h.float().to(A_r.device) - mx; sh = (xc @ A_r) @ D_r.T
        return sh, xc - sh
    test = torch.randn(4, D_MODEL, device=A_r.device) * 3 + mx
    sh, _ = split(test); sh2, _ = split(sh + mx)
    print(f"  layer {l}: shared dims {len(idx)} of {D_MODEL}; oblique projection idempotence error {float((sh2 - sh).norm() / sh.norm()):.1e}")
    return split, mx

def ablate_fn(split, mx, which):
    def fn(h):
        sh, pr = split(h); dev = h.device
        if which == "private": return (mx + sh).to(dev)
        if which == "shared": return (mx + pr).to(dev)
        return mx.expand_as(sh).to(dev)
    return fn

def interchange_fn(split, mx, donor, which):
    def fn(h):
        sh, pr = split(h); dsh, dpr = split(donor); xc = sh + pr
        if which == "private": new = xc - pr + dpr
        elif which == "shared": new = xc - sh + dsh
        else: new = dsh + dpr
        return (new + mx).to(h.device)
    return fn

if RUN_D and "es" in C_RES and ladder and not have("D_mediation.json"):
    D_out = {}
    med_layers = sorted({l for (lg, l) in C_FITS if lg == "es"})
    for l in med_layers:
        with Timer(f"Part D: layer {l}"):
            split, mx = decomposer("es", l)
            # --- 1. ablation dissociation ---
            es_sents = [S for S in SENTENCES if S["lang"] == "es"]
            targets = []
            for S in es_sents:
                hi = hot_word_index(S)
                for k, (w, a, b) in enumerate(words_with_spans(S["text"])):
                    if k in hi or (len(w) >= 4 and S["cat"] == "control"): targets.append((S, w, k in hi, (a, b)))
            ask = [chat_prompt(f'{S["text"]}\n\nIn the Spanish sentence above, what does the word "{w}" mean here? Answer in one line.')
                   for S, w, _, _ in targets]
            expl = generate(ask, max_new_tokens=40)
            P, Cn, E, meta = [], [], [], []
            for (S, w, is_hot, span), ap, ex in zip(targets, ask, expl):
                tp = chat_prompt(rung_user("B1", S["text"], "es"))
                for task, prompt, cont in [("translate", tp, ladder[S["id"]]["B1"]), ("explain", ap, ex)]:
                    pos = positions_in_prompt(prompt, S["text"], span=span)
                    if task == "explain":                      # also the word quoted in the question
                        q = prompt.rfind(f'"{w}"') + 1; pos = sorted(set(pos) | set(positions_at(prompt, q, q + len(w))))
                    for cond in ["none", "private", "shared", "full"]:
                        P.append(prompt); Cn.append(cont)
                        E.append(None if cond == "none" else [(l, pos, ablate_fn(split, mx, cond))])
                        meta.append((S["id"], w, is_hot, task, cond))
            sc = score(P, Cn, edits=E)
            lp = {m: float(r["lp"].mean()) for m, r in zip(meta, sc)}
            abl = []
            for (S, w, is_hot, _), ex in zip(targets, expl):
                row = dict(sid=S["id"], word=w, hot=is_hot, explanation=ex)
                for task in ["translate", "explain"]:
                    base = lp[(S["id"], w, is_hot, task, "none")]
                    for cond in ["private", "shared", "full"]:
                        row[f"{task}_{cond}"] = base - lp[(S["id"], w, is_hot, task, cond)]   # drop in log-prob per token
                abl.append(row)
            # --- 2. interchange on sense pairs ---
            inter = []
            for SP in SENSE_PAIRS:
                runs = {}
                for me, other in [("a", "b"), ("b", "a")]:
                    pm = chat_prompt(rung_user("B1", SP[me], "es")); po = chat_prompt(rung_user("B1", SP[other], "es"))
                    pos_m = positions_in_prompt(pm, SP[me], SP["word"]); pos_o = positions_in_prompt(po, SP[other], SP["word"])
                    if len(pos_m) != len(pos_o): print(f"  skip {SP['id']}: token counts differ"); runs = None; break
                    donor = capture_prompt_states(po, l)[pos_o]
                    tmpl = SP[f"en_{me}"]; cands = [tmpl.format(SP["sense_a"]), tmpl.format(SP["sense_b"])]
                    conds = {"none": None, "private": interchange_fn(split, mx, donor, "private"),
                             "shared": interchange_fn(split, mx, donor, "shared"), "full": interchange_fn(split, mx, donor, "full")}
                    Ps, Cs, Es, keys = [], [], [], []
                    for cname, fn in conds.items():
                        for ci, c in enumerate(cands):
                            Ps.append(pm); Cs.append(c); Es.append(None if fn is None else [(l, pos_m, fn)]); keys.append((cname, ci))
                    r = score(Ps, Cs, edits=Es); tot = {k: float(x["lp"].sum()) for k, x in zip(keys, r)}
                    lo = {c: tot[(c, 0)] - tot[(c, 1)] for c in conds}          # log-odds sense_a vs sense_b
                    gen = {c: generate_with_edits(pm, [] if fn is None else [(l, pos_m, fn)]) for c, fn in conds.items()}
                    runs[me] = dict(logodds=lo, generations=gen)
                if runs is None: continue
                base_a, base_b = runs["a"]["logodds"]["none"], runs["b"]["logodds"]["none"]
                gap = base_b - base_a
                flip = {}
                for c in ["private", "shared", "full"]:
                    fa = (runs["a"]["logodds"][c] - base_a) / gap if abs(gap) > 1e-6 else float("nan")
                    fb = (runs["b"]["logodds"][c] - base_b) / (-gap) if abs(gap) > 1e-6 else float("nan")
                    flip[c] = float(np.nanmean([fa, fb]))
                inter.append(dict(id=SP["id"], base_gap=gap, flip=flip, runs=runs))
                print(f"  {SP['id']:11s} base log-odds a={base_a:+.1f} b={base_b:+.1f}  flip private={flip['private']:+.2f} "
                      f"shared={flip['shared']:+.2f} full={flip['full']:+.2f}")
            D_out[l] = dict(ablation=abl, interchange=inter)
    save_json("D_mediation.json", D_out)

In [ ]:
D = load_json("D_mediation.json") if have("D_mediation.json") else None
if D:
    for l, d in D.items():
        ab = d["ablation"]
        def ratio(r, task, cond): f = r[f"{task}_full"]; return r[f"{task}_{cond}"] / f if abs(f) > 1e-6 else np.nan
        print(f"\nlayer {l}: effect of removing the PRIVATE part, as a share of removing the whole state (mean, 95% CI)")
        for lab, grp in [("hot words", [r for r in ab if r["hot"]]), ("control words", [r for r in ab if not r["hot"]])]:
            t = [ratio(r, "translate", "private") for r in grp]; e = [ratio(r, "explain", "private") for r in grp]
            ct, ce = boot_ci(t), boot_ci(e)
            print(f"  {lab:14s} n={len(grp):3d}  on translation {np.nanmean(t):.2f} [{ct[0]:.2f}, {ct[1]:.2f}] (median {np.nanmedian(t):.2f})"
                  f"   on explanation {np.nanmean(e):.2f} [{ce[0]:.2f}, {ce[1]:.2f}] (median {np.nanmedian(e):.2f})")
        hot = [r for r in ab if r["hot"]]
        gap = [ratio(r, "explain", "private") - ratio(r, "translate", "private") for r in hot]
        cg = boot_ci(gap)
        print(f"  hot words: explanation minus translation = {np.nanmean(gap):+.2f} [{cg[0]:+.2f}, {cg[1]:+.2f}] (median {np.nanmedian(gap):+.2f}); positive = understood but not translated")
    l = list(D)[0]; ab = D[l]["ablation"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    for grp, col, lab in [([r for r in ab if not r["hot"]], EASY, "ordinary words"), ([r for r in ab if r["hot"]], HARD, "hard words")]:
        x = [r["translate_private"] / r["translate_full"] if abs(r["translate_full"]) > 1e-6 else np.nan for r in grp]
        y = [r["explain_private"] / r["explain_full"] if abs(r["explain_full"]) > 1e-6 else np.nan for r in grp]
        x, y = np.clip(x, -0.5, 1.5), np.clip(y, -0.5, 1.5)     # extreme ratios (tiny denominators) pinned to the edge
        ax.scatter(x, y, color=col, s=28, alpha=0.8, label=lab)
        for r, xi, yi in zip(grp, x, y):
            if r["hot"]: ax.text(xi, yi, " " + r["word"], fontsize=7, style="italic", color=col)
    ax.plot([-1, 2], [-1, 2], color="#bbb", lw=0.8); ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("harm to the TRANSLATION\n(removing the language-private part, share of removing everything)")
    ax.set_ylabel("harm to EXPLAINING the word")
    ax.set_title("Above the diagonal: the model understands more than it translates", loc="left", fontsize=9); ax.legend(frameon=False, fontsize=8)
    ax = axes[1]; it = D[l]["interchange"]; w = 0.26
    for k, (c, col) in enumerate([("private", HARD), ("shared", EASY), ("full", GREY)]):
        ax.bar(np.arange(len(it)) + (k - 1) * w, [x["flip"][c] for x in it], w, color=col, label=f"swap the {c} part" if c != "full" else "swap everything")
    ax.set_xticks(np.arange(len(it))); ax.set_xticklabels([x["id"] for x in it], rotation=30, ha="right", style="italic")
    ax.axhline(0, color="#888", lw=0.8); ax.axhline(1, color="#888", lw=0.8, ls=":")
    ax.set_ylabel("how far the English moves to the other sense"); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Swapping part of one word's state between two senses", loc="left", fontsize=9)
    savefig(fig, f"D_mediation_L{l}")
    print("\nExample generations under interchange (frame a, patched from frame b):")
    for x in D[l]["interchange"][:8]:
        g = x["runs"]["a"]["generations"]; print(f"  {x['id']:11s} none: {g['none'][:60]!r} | private: {g['private'][:60]!r} | full: {g['full'][:60]!r}")

## Summary
Collects the headline numbers into `SUMMARY.md` in the Drive folder. Read it alongside the figures in `figures/`.

In [ ]:
lines = [f"# Summary: {MODEL_ID}", "", f"{N_LAYERS} layers, d = {D_MODEL}. Every number is inside this one model.", ""]
if A:
    lines += ["## A. English detour", f"- English peaks at layer {A['detour_peak_layer']}/{N_LAYERS}; Italian overtakes at {A['italian_overtakes_at']}.",
              f"- Peak mean P(English) = {np.array(A['P_en']).mean(0).max():.3f} over {len(A['P_en'])} words.", ""]
if have("B_summary.json"):
    bs = load_json("B_summary.json"); ps = bs["per_sentence"]
    lines += ["## B. Round trip (masked)", f"- Tolerance ε = {bs['epsilon']:.3f}; hot-word AUROC = {bs['auroc_hot_words']:.3f}."]
    for c in ["untranslatable", "construction", "idiom", "control"]:
        v = [x for x in ps.values() if x["cat"] == c]
        if v: lines.append(f"- {c}: U = {np.mean([x['U'] for x in v]):.2f} doublings, borrowing buys {np.nanmean([x['borrow_gain'] for x in v]):.3f} (masked: {np.nanmean([x['borrow_gain_masked'] for x in v]):.3f}).")
    fa = B["duende"]; wl = dict(zip(fa["words"], fa["rungs"]["B1"]["word_left_masked"]))
    lines += [f"- *Me gusta mucho el duende* → {ladder['duende']['B1']!r}; leftover duende {wl.get('duende', np.nan):.2f}, gusta {wl.get('gusta', np.nan):.2f}, mucho {wl.get('mucho', np.nan):.2f}.",
              "- Sense recovered for *duende* (standard / keep / gloss / footnote / thick): " + " / ".join(f"{fa['sense']['duende']['norm'][r]:.2f}" for r in ["B1", "B2", "B3", "B5", "B6"]), ""]
for lang in C_RES:
    r = next(x for x in C_RES[lang] if x["layer"] == BEST_LAYER[lang])
    lines += [f"## C. Spectrum {lang}–en (layer {r['layer']})",
              f"- {r['n_shared']} shared directions above the null (of {D_MODEL}); fit on {r['n_fit_pairs']} word pairs.",
              f"- English predicted from {lang}: {r['E_en_from_x']:.3f}; {lang} predicted from English: {r['E_x_from_en']:.3f}; bias {r['bias']:+.3f} (standardised {r['bias_std']:+.3f})."]
    if C_SPANS.get(lang):
        au = C_SPANS[lang][BEST_LAYER[lang]]["auroc"]; ag = C_SPANS[lang][BEST_LAYER[lang]]["spearman_with_roundtrip_A"]
        lines.append(f"- AUROC hot vs other words: U {au['U']:.2f}, U_raw {au['U_raw']:.2f}, #tokens baseline {au['ntok']:.2f}; agreement with round trip {ag}.")
    lines.append("")
if D:
    for l, d in D.items():
        ab = d["ablation"]; hot = [r for r in ab if r["hot"]]
        f = lambda r, t: r[f"{t}_private"] / r[f"{t}_full"] if abs(r[f"{t}_full"]) > 1e-6 else np.nan
        lines += [f"## D. Mediation (layer {l})",
                  f"- Hot words, private-part share of harm: translation {np.nanmean([f(r, 'translate') for r in hot]):.2f}, explanation {np.nanmean([f(r, 'explain') for r in hot]):.2f}.",
                  "- Flip fractions (private / shared / full): " + "; ".join(f"{x['id']} {x['flip']['private']:+.2f}/{x['flip']['shared']:+.2f}/{x['flip']['full']:+.2f}" for x in d["interchange"]), ""]
open(os.path.join(OUT, "SUMMARY.md"), "w").write("\n".join(lines))
print("\n".join(lines))